* pip install pydub PyOpenGL PyOpenGL_accelerate glfw pillow freetype-py pygame PyOpenGL PyOpenGL_accelerate numpy imageio_ffmpeg
* apt update && apt install -y libgl1-mesa-glx libgl1-mesa-dri libglu1-mesa freeglut3-dev libglfw3 libglfw3-dev libx11-dev
* apt install -y libxrandr-dev libxinerama-dev libxcursor-dev libxi-dev
* apt install -y ffmpeg
* Fuera del contenedor de docker ejecutar "xhost +local:docker" o "xhost +" para que pueda acceder al display


## Codigos

In [ ]:
#Despliega una ventana donde se ve una espiral detrás. Además va creando una imagen por cada frame. Es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo = """
import glfw
from OpenGL.GL import *
from OpenGL.GLUT import *
from PIL import Image
import time
import math
import os

FRAME_DIR = "frames"
os.makedirs(FRAME_DIR, exist_ok=True)

def init_window(width, height, title):
    if not glfw.init():
        raise Exception("GLFW no pudo iniciarse")
    glfw.window_hint(glfw.RESIZABLE, False)
    window = glfw.create_window(width, height, title, None, None)
    if not window:
        glfw.terminate()
        raise Exception("No se pudo crear la ventana GLFW")
    glfw.make_context_current(window)
    return window

def draw_spiral(t):
    glBegin(GL_LINE_STRIP)
    num_turns = 10
    points = 1000
    for i in range(points):
        theta = 2 * math.pi * num_turns * i / points + t
        r = i / points
        x = r * math.cos(theta)
        y = r * math.sin(theta)
        glColor3f(0.5 + 0.5 * math.cos(t), 0.5 + 0.5 * math.sin(t), 0.7)
        glVertex2f(x, y)
    glEnd()

def draw_text(text, x, y, scale, r, g, b):
    glPushMatrix()
    glTranslatef(x, y, 0)
    glScalef(scale, scale, 1)
    glColor3f(r, g, b)
    for ch in text:
        glutStrokeCharacter(GLUT_STROKE_ROMAN, ord(ch))
    glPopMatrix()

def main():
    glutInit()
    window = init_window(1920, 1080, "Hipnosis con frase")
    glClearColor(0, 0, 0, 1)
    glLineWidth(2)

    frame_count = 0
    while not glfw.window_should_close(window) and frame_count < 300:
        t = time.time()
        glClear(GL_COLOR_BUFFER_BIT)
        glLoadIdentity()

        draw_spiral(t)
        pulsate = 0.001 + 0.0008 * math.sin(t * 2)
        color_shift = (math.sin(t), math.cos(t), math.sin(t + 2))
        draw_text("Confía en mí...", -0.4, -0.1, pulsate * 500, *color_shift)

        # Captura de fotograma
        width, height = glfw.get_framebuffer_size(window)
        glPixelStorei(GL_PACK_ALIGNMENT, 1)
        data = glReadPixels(0, 0, width, height, GL_RGB, GL_UNSIGNED_BYTE)
        from PIL import Image
        image = Image.frombytes("RGB", (width, height), data)
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        image.save(f"{FRAME_DIR}/frame_{frame_count:04d}.png")

        glfw.swap_buffers(window)
        glfw.poll_events()
        frame_count += 1

    glfw.terminate()

if __name__ == "__main__":
    main()
    
"""

In [ ]:
#Despliega una ventana donde se muestra un texto creado con vectores y se ve una espiral detrás. Además va creando una imagen por cada frame. Es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo = """
import glfw
from OpenGL.GL import *
from OpenGL.GLUT import *
from PIL import Image
import time
import math
import os

FRAME_DIR = "frames"
os.makedirs(FRAME_DIR, exist_ok=True)

def init_window(width, height, title):
    if not glfw.init():
        raise Exception("GLFW no pudo iniciarse")
    glfw.window_hint(glfw.RESIZABLE, False)
    window = glfw.create_window(width, height, title, None, None)
    if not window:
        glfw.terminate()
        raise Exception("No se pudo crear la ventana GLFW")
    glfw.make_context_current(window)
    return window

def draw_spiral(t):
    glBegin(GL_LINE_STRIP)
    num_turns = 10
    points = 1000
    for i in range(points):
        theta = 2 * math.pi * num_turns * i / points + t
        r = i / points
        x = r * math.cos(theta)
        y = r * math.sin(theta)
        glColor3f(0.5 + 0.5 * math.cos(t), 0.5 + 0.5 * math.sin(t), 0.7)
        glVertex2f(x, y)
    glEnd()

def draw_text_centered(text, scale, r, g, b):
    # Medimos el ancho del texto para centrarlo
    total_width = sum([glutStrokeWidth(GLUT_STROKE_ROMAN, ord(c)) for c in text])
    glPushMatrix()
    glTranslatef(-total_width * scale / 2 / 500.0, -scale / 2, 0)
    glScalef(scale / 100.0, scale / 100.0, 1)
    glColor3f(r, g, b)
    for ch in text:
        glutStrokeCharacter(GLUT_STROKE_ROMAN, ord(ch))
    glPopMatrix()

def main():
    glutInit()
    window = init_window(800, 800, "Hipnosis con texto animado")
    glClearColor(0, 0, 0, 1)
    glLineWidth(2)

    frame_count = 0
    while not glfw.window_should_close(window) and frame_count < 300:
        t = time.time()
        glClear(GL_COLOR_BUFFER_BIT)
        glLoadIdentity()

        draw_spiral(t)

        # Animación de texto: escala pulsante y color cambiante
        scale = 3.0 + math.sin(t * 2) * 0.5
        r = 0.5 + 0.5 * math.sin(t)
        g = 0.5 + 0.5 * math.cos(t * 0.7)
        b = 0.5 + 0.5 * math.sin(t * 1.3)

        draw_text_centered("Confía en mí...", scale, r, g, b)

        # Captura del fotograma
        width, height = glfw.get_framebuffer_size(window)
        glPixelStorei(GL_PACK_ALIGNMENT, 1)
        data = glReadPixels(0, 0, width, height, GL_RGB, GL_UNSIGNED_BYTE)
        image = Image.frombytes("RGB", (width, height), data)
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        image.save(f"{FRAME_DIR}/frame_{frame_count:04d}.png")

        glfw.swap_buffers(window)
        glfw.poll_events()
        frame_count += 1

    glfw.terminate()

if __name__ == "__main__":
    main()
"""

In [ ]:
#Despliega una ventana donde se va mostrando texto y se ve una espiral detrás. Además va creando una imagen por cada frame. Es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo = """
import glfw
from OpenGL.GL import *
from PIL import Image, ImageDraw, ImageFont
import time
import math
import numpy as np
import os

FRAME_DIR = "frames"
os.makedirs(FRAME_DIR, exist_ok=True)

FONT_PATH = "DejaVuSans.ttf"  # Asegúrate de que este archivo esté en el mismo directorio

def init_window(width, height, title):
    if not glfw.init():
        raise Exception("GLFW no pudo iniciarse")
    glfw.window_hint(glfw.RESIZABLE, False)
    window = glfw.create_window(width, height, title, None, None)
    if not window:
        glfw.terminate()
        raise Exception("No se pudo crear la ventana GLFW")
    glfw.make_context_current(window)
    return window

def create_text_texture(text, font_size):
    font = ImageFont.truetype(FONT_PATH, font_size)
    img_size = (1024, 256)
    image = Image.new("RGBA", img_size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(image)

    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]
    position = ((img_size[0] - text_width) // 2, (img_size[1] - text_height) // 2)
    draw.text(position, text, font=font, fill=(255, 255, 255, 255))

    image = image.transpose(Image.FLIP_TOP_BOTTOM)
    image_data = np.array(image, dtype=np.uint8)

    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, image.width, image.height, 0,
                 GL_RGBA, GL_UNSIGNED_BYTE, image_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    return tex_id, image.width, image.height

def draw_texture(tex_id, w, h, scale):
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glColor3f(1, 1, 1)

    sw = w / 800 * scale
    sh = h / 800 * scale

    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex2f(-sw, -sh)
    glTexCoord2f(1, 0); glVertex2f(sw, -sh)
    glTexCoord2f(1, 1); glVertex2f(sw, sh)
    glTexCoord2f(0, 1); glVertex2f(-sw, sh)
    glEnd()

    glDisable(GL_TEXTURE_2D)

def draw_spiral(t):
    glBegin(GL_LINE_STRIP)
    num_turns = 10
    points = 1000
    for i in range(points):
        theta = 2 * math.pi * num_turns * i / points + t
        r = i / points
        x = r * math.cos(theta)
        y = r * math.sin(theta)
        glColor3f(0.5 + 0.5 * math.cos(t), 0.5 + 0.5 * math.sin(t), 0.7)
        glVertex2f(x, y)
    glEnd()

def main():
    window = init_window(800, 800, "Hipnosis con fuente real")
    glClearColor(0, 0, 0, 1)
    glEnable(GL_BLEND)
    glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)

    tex_id, tex_w, tex_h = create_text_texture("Hola, yo soy El Carretero, y en mis multiples viajes he visto muchas cosas, y he escuchado infinidad de historias... hoy te contaré una a la que llamo: La niebla todavía habla.", 64)

    frame_count = 0
    while not glfw.window_should_close(window) and frame_count < 300:
        t = time.time()
        glClear(GL_COLOR_BUFFER_BIT)
        glLoadIdentity()

        draw_spiral(t)

        # Animación: escala pulsante
        scale = 1.0 + 0.2 * math.sin(t * 2)
        draw_texture(tex_id, tex_w, tex_h, scale)

        # Captura de fotograma
        width, height = glfw.get_framebuffer_size(window)
        glPixelStorei(GL_PACK_ALIGNMENT, 1)
        data = glReadPixels(0, 0, width, height, GL_RGB, GL_UNSIGNED_BYTE)
        image = Image.frombytes("RGB", (width, height), data)
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        image.save(f"{FRAME_DIR}/frame_{frame_count:04d}.png")

        glfw.swap_buffers(window)
        glfw.poll_events()
        frame_count += 1

    glfw.terminate()

if __name__ == "__main__":
    main()
"""

In [ ]:
#Despliega una ventana donde se va mostrando texto y se ve una espiral detrás. Además va creando una imagen por cada frame. Es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo = """
import glfw
from OpenGL.GL import *
from PIL import Image, ImageDraw, ImageFont
import time
import math
import numpy as np
import os

# Ajustes
FRAME_DIR = "frames"
FONT_PATH = "DejaVuSans.ttf"
os.makedirs(FRAME_DIR, exist_ok=True)

WIDTH, HEIGHT = 1920, 1080
# WIDTH, HEIGHT = 1080,1920
# textos = ["Confía en mí...", "Relájate", "Respira hondo"]
# duraciones = [3.0, 2.5, 4.0]

textos = ['Hola', ' yo soy El Carretero', ' y en mis multiples viajes he visto muchas cosas', ' y he escuchado infinidad de historias', ' hoy te contaré una a la que llamo', ' La niebla todavía habla']
duraciones = [0.28125, 1.40625, 2.8125, 1.96875, 2.53125, 1.40625]

typing_speed = 0.05  # segundos entre letras

def init_window(width, height, title):
    if not glfw.init():
        raise Exception("GLFW no pudo iniciarse")
    glfw.window_hint(glfw.RESIZABLE, False)
    window = glfw.create_window(width, height, title, None, None)
    if not window:
        glfw.terminate()
        raise Exception("No se pudo crear la ventana GLFW")
    glfw.make_context_current(window)
    return window

def fit_font_size(text, max_width, max_height, min_size=10, max_size=100):
    for size in range(max_size, min_size, -1):
        font = ImageFont.truetype(FONT_PATH, size)
        dummy_img = Image.new("RGB", (1,1))
        draw = ImageDraw.Draw(dummy_img)
        bbox = draw.textbbox((0,0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]
        if text_w <= max_width and text_h <= max_height:
            return font
    return ImageFont.truetype(FONT_PATH, min_size)

def create_text_texture(text, alpha):
    margin = 50
    img_size = (WIDTH, 256)
    font = fit_font_size(text, img_size[0] - 2*margin, img_size[1] - 2*margin)
    image = Image.new("RGBA", img_size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(image)
    bbox = draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    position = ((img_size[0] - text_w) // 2, (img_size[1] - text_h) // 2)
    draw.text(position, text, font=font, fill=(255, 255, 255, int(alpha * 255)))
    image = image.transpose(Image.FLIP_TOP_BOTTOM)
    image_data = np.array(image, dtype=np.uint8)

    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, image.width, image.height, 0,
                 GL_RGBA, GL_UNSIGNED_BYTE, image_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    return tex_id, image.width, image.height

def draw_texture(tex_id, w, h, scale):
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glColor3f(1, 1, 1)
    sw = w / WIDTH * scale
    sh = h / WIDTH * scale
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex2f(-sw, -sh)
    glTexCoord2f(1, 0); glVertex2f(sw, -sh)
    glTexCoord2f(1, 1); glVertex2f(sw, sh)
    glTexCoord2f(0, 1); glVertex2f(-sw, sh)
    glEnd()
    glDisable(GL_TEXTURE_2D)

def draw_spiral(t):
    glBegin(GL_LINE_STRIP)
    num_turns = 10
    points = 1000
    for i in range(points):
        theta = 2 * math.pi * num_turns * i / points + t
        r = i / points
        x = r * math.cos(theta)
        y = r * math.sin(theta)
        glColor3f(0.5 + 0.5 * math.cos(t), 0.5 + 0.5 * math.sin(t), 0.7)
        glVertex2f(x, y)
    glEnd()

def main():
    window = init_window(WIDTH, HEIGHT, "Hipnosis animada")
    glClearColor(0, 0, 0, 1)
    glEnable(GL_BLEND)
    glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)

    current_index = 0
    start_time = time.time()
    text = textos[current_index]
    duration = duraciones[current_index]
    frame_count = 0

    while not glfw.window_should_close(window) and current_index < len(textos):
        now = time.time()
        elapsed = now - start_time

        glClear(GL_COLOR_BUFFER_BIT)
        glLoadIdentity()

        draw_spiral(now)

        # Efecto máquina de escribir
        total_letters = min(len(text), int(elapsed / typing_speed))
        visible_text = text[:total_letters]

        # Fade out
        if elapsed > duration:
            fade = max(0.0, 1.0 - (elapsed - duration) / 1.0)
        else:
            fade = 1.0

        if visible_text:
            tex_id, tex_w, tex_h = create_text_texture(visible_text, fade)
            scale = 1.0 + 0.02 * math.sin(now * 2)
            draw_texture(tex_id, tex_w, tex_h, scale)
            glDeleteTextures(1, [tex_id])

        # Avanzar al siguiente texto
        if elapsed > duration + 1.0:
            current_index += 1
            if current_index < len(textos):
                text = textos[current_index]
                duration = duraciones[current_index]
                start_time = now

        # Captura de fotograma
        width, height = glfw.get_framebuffer_size(window)
        glPixelStorei(GL_PACK_ALIGNMENT, 1)
        data = glReadPixels(0, 0, width, height, GL_RGB, GL_UNSIGNED_BYTE)
        image = Image.frombytes("RGB", (width, height), data)
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        image.save(f"{FRAME_DIR}/frame_{frame_count:04d}.png")

        glfw.swap_buffers(window)
        glfw.poll_events()
        frame_count += 1

    glfw.terminate()

if __name__ == "__main__":
    main()
"""

In [33]:
#Despliega una ventana donde se va mostrando texto y se ven particulas detrás. Además va creando una imagen por cada frame. Es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo = """
import glfw
from OpenGL.GL import *
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import time
import math
import numpy as np
import os
import random

# Config
# WIDTH, HEIGHT = 800, 800
WIDTH, HEIGHT = 1920, 1080
FONT_PATH = "DejaVuSans.ttf"
FRAME_DIR = "frames"
os.makedirs(FRAME_DIR, exist_ok=True)

textos = ['Hola', ' yo soy El Carretero', ' y en mis multiples viajes he visto muchas cosas', ' y he escuchado infinidad de historias', ' hoy te contaré una a la que llamo', ' La niebla todavía habla']
duraciones = [0.28125, 1.40625, 2.8125, 1.96875, 2.53125, 1.40625]
typing_speed = 0.05

def init_window(width, height, title):
    if not glfw.init():
        raise Exception("GLFW no pudo iniciarse")
    glfw.window_hint(glfw.RESIZABLE, False)
    window = glfw.create_window(width, height, title, None, None)
    if not window:
        glfw.terminate()
        raise Exception("No se pudo crear la ventana GLFW")
    glfw.make_context_current(window)
    return window

def fit_font_size(text, max_width, max_height, min_size=10, max_size=100):
    for size in range(max_size, min_size, -1):
        font = ImageFont.truetype(FONT_PATH, size)
        img = Image.new("RGBA", (1,1))
        draw = ImageDraw.Draw(img)
        bbox = draw.textbbox((0,0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]
        if text_w <= max_width and text_h <= max_height:
            return font
    return ImageFont.truetype(FONT_PATH, min_size)

def create_text_texture(text, alpha):
    size = (WIDTH, HEIGHT)
    margin = 50
    font = fit_font_size(text, size[0]-2*margin, size[1]//5)
    base = Image.new("RGBA", size, (0,0,0,0))
    draw = ImageDraw.Draw(base)

    bbox = draw.textbbox((0,0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    position = ((size[0] - text_w) // 2, (size[1] - text_h) // 2)

    # Sombra blanca
    shadow1 = Image.new("RGBA", size, (0,0,0,0))
    draw1 = ImageDraw.Draw(shadow1)
    draw1.text(position, text, font=font, fill=(255,255,255,int(alpha*128)))
    shadow1 = shadow1.filter(ImageFilter.GaussianBlur(6))
    
    # Sombra azul
    shadow2 = Image.new("RGBA", size, (0,0,0,0))
    draw2 = ImageDraw.Draw(shadow2)
    draw2.text(position, text, font=font, fill=(100,100,255,int(alpha*128)))
    shadow2 = shadow2.filter(ImageFilter.GaussianBlur(12))

    # Texto principal
    draw.text(position, text, font=font, fill=(255,255,255,int(alpha*255)))

    result = Image.alpha_composite(Image.alpha_composite(shadow2, shadow1), base)
    result = result.transpose(Image.FLIP_TOP_BOTTOM)
    image_data = np.array(result, dtype=np.uint8)

    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, size[0], size[1], 0,
                 GL_RGBA, GL_UNSIGNED_BYTE, image_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    return tex_id

def draw_texture(tex_id):
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glColor3f(1,1,1)
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex2f(-1, -1)
    glTexCoord2f(1, 0); glVertex2f(1, -1)
    glTexCoord2f(1, 1); glVertex2f(1, 1)
    glTexCoord2f(0, 1); glVertex2f(-1, 1)
    glEnd()
    glDisable(GL_TEXTURE_2D)

def hsv_to_rgb(h, s, v):
    h = h % 1
    i = int(h*6)
    f = h*6 - i
    p = v*(1 - s)
    q = v*(1 - f*s)
    t = v*(1 - (1 - f)*s)
    i = i % 6
    if i == 0: return (v, t, p)
    if i == 1: return (q, v, p)
    if i == 2: return (p, v, t)
    if i == 3: return (p, q, v)
    if i == 4: return (t, p, v)
    if i == 5: return (v, p, q)

class Particle:
    def __init__(self, angle, speed):
        self.angle = angle
        self.speed = speed
        self.radius = 0
        self.alpha = 1.0

    def update(self, dt):
        self.radius += self.speed * dt
        self.alpha -= dt * 0.5

    def draw(self, t, color):
        if self.alpha <= 0: return
        x = self.radius * math.cos(self.angle)
        y = self.radius * math.sin(self.angle)
        r, g, b = color
        glColor4f(r, g, b, self.alpha)
        glPointSize(2.5)
        glBegin(GL_POINTS)
        glVertex2f(x, y)
        glEnd()

def draw_starburst(particles, dt, t):
    hue = (t * 0.02) % 1.0
    color = hsv_to_rgb(hue, 0.7, 1.0)
    for p in particles:
        p.update(dt)
        p.draw(t, color)
    # Generar nuevas partículas
    if len(particles) < 100:
        for _ in range(3):
            particles.append(Particle(random.uniform(0, 2*math.pi), random.uniform(0.1, 0.5)))

def main():
    window = init_window(WIDTH, HEIGHT, "Hipnosis con estrella fugaz")
    glClearColor(0,0,0,1)
    glEnable(GL_BLEND)
    glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)

    current_index = 0
    start_time = time.time()
    text = textos[current_index]
    duration = duraciones[current_index]
    frame_count = 0
    particles = []

    while not glfw.window_should_close(window) and current_index < len(textos):
        now = time.time()
        elapsed = now - start_time

        glClear(GL_COLOR_BUFFER_BIT)
        glLoadIdentity()

        # Estrella fugaz animada
        draw_starburst(particles, 1/60.0, now)

        # Efecto máquina de escribir
        total_letters = min(len(text), int(elapsed / typing_speed))
        visible_text = text[:total_letters]

        if elapsed > duration:
            fade = max(0.0, 1.0 - (elapsed - duration) / 1.0)
        else:
            fade = 1.0

        if visible_text:
            tex_id = create_text_texture(visible_text, fade)
            draw_texture(tex_id)
            glDeleteTextures(1, [tex_id])

        if elapsed > duration + 1.0:
            current_index += 1
            if current_index < len(textos):
                text = textos[current_index]
                duration = duraciones[current_index]
                start_time = now

        width, height = glfw.get_framebuffer_size(window)
        glPixelStorei(GL_PACK_ALIGNMENT, 1)
        data = glReadPixels(0, 0, width, height, GL_RGB, GL_UNSIGNED_BYTE)
        image = Image.frombytes("RGB", (width, height), data)
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        image.save(f"{FRAME_DIR}/frame_{frame_count:04d}.png")
        frame_count += 1

        glfw.swap_buffers(window)
        glfw.poll_events()

    glfw.terminate()

if __name__ == "__main__":
    main()

"""

In [ ]:
#Despliega una ventana con una estrella fugaz entre muchas comillas de un lado a otro. es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo = """
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import random
import time
import numpy as np

# Configuración de la ventana
width, height = 1920, 1080
pygame.init()
screen = pygame.display.set_mode((width, height), DOUBLEBUF | OPENGL)
pygame.display.set_caption("Estrella Fugaz con Partículas")

# Inicialización de OpenGL
glViewport(0, 0, width, height)
glMatrixMode(GL_PROJECTION)
glLoadIdentity()
gluPerspective(45, (width / height), 0.1, 100.0)
glMatrixMode(GL_MODELVIEW)
glLoadIdentity()
glTranslatef(0.0, 0.0, -10.0)
glClearColor(0.0, 0.0, 0.1, 1.0)  # Fondo oscuro
glEnable(GL_DEPTH_TEST)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_POINT_SMOOTH)
glPointSize(2.0)

# Datos de la estrella fugaz
star_position = np.array([-5.0, 5.0, 0.0], dtype=np.float32)
star_velocity = np.array([0.05, -0.03, 0.0], dtype=np.float32)
trail_positions = []
max_trail_length = 50

# Datos de las partículas
particles = []
max_particles = 5000
particle_lifetime = 60  # Número de frames que vive una partícula
particle_speed = 0.02
particle_size = 1.0

def generate_particle(position):
    direction = np.random.uniform(-1, 1, 3).astype(np.float32)
    direction = direction / np.linalg.norm(direction) * particle_speed
    color = np.random.uniform(0.5, 1.0, 4).astype(np.float32)
    color[3] = 1.0  # Alpha
    return {
        'position': np.array(position, dtype=np.float32),
        'velocity': direction,
        'lifetime': particle_lifetime,
        'color': color
    }

def update_star():
    global star_position, trail_positions
    star_position += star_velocity

    trail_positions.append(star_position.copy())
    if len(trail_positions) > max_trail_length:
        trail_positions.pop(0)

    # Generar nuevas partículas
    num_new_particles = random.randint(5, 15)
    for _ in range(num_new_particles):
        particles.append(generate_particle(star_position))

def update_particles():
    global particles
    new_particles = []
    for p in particles:
        p['position'] += p['velocity']
        p['lifetime'] -= 1
        if p['lifetime'] > 0:
            new_particles.append(p)
    particles = new_particles

def render_star():
    glColor3f(1.0, 1.0, 0.8)  # Color amarillo claro para la estrella
    glPointSize(5.0)
    glBegin(GL_POINTS)
    glVertex3fv(star_position)
    glEnd()
    glPointSize(2.0) # Restaurar tamaño de punto

def render_trail():
    glBegin(GL_LINE_STRIP)
    glColor4f(1.0, 1.0, 0.8, 0.5) # Color del rastro con transparencia
    for pos in trail_positions:
        glVertex3fv(pos)
    glEnd()

def render_particles():
    glBegin(GL_POINTS)
    for p in particles:
        glColor4fv(p['color'])
        glVertex3fv(p['position'])
    glEnd()

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glLoadIdentity()
    glTranslatef(0.0, 0.0, -10.0)

    update_star()
    update_particles()

    render_star()
    render_trail()
    render_particles()

    pygame.display.flip()
    time.sleep(0.01)

pygame.quit()
"""

In [ ]:
#Despliega una ventana con una estrella fugaz entre muchas comillas. es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo = """
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import random
import time
import numpy as np
from OpenGL.GLUT import glutSolidSphere, glutInit

# Configuración de la ventana
width, height = 1920, 1080
pygame.init()
screen = pygame.display.set_mode((width, height), DOUBLEBUF | OPENGL)
pygame.display.set_caption("Estrella Fugaz 3D con Rotación por Teclado")

# Inicializar GLUT
glutInit()

# Inicialización de OpenGL
glViewport(0, 0, width, height)
glMatrixMode(GL_PROJECTION)
glLoadIdentity()
gluPerspective(45, (width / height), 0.1, 100.0)
glMatrixMode(GL_MODELVIEW)
glLoadIdentity()
glTranslatef(0.0, 0.0, -10.0)
glClearColor(0.0, 0.0, 0.1, 1.0)
glEnable(GL_DEPTH_TEST)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_POINT_SMOOTH)
glPointSize(2.0)

# Datos de la estrella fugaz (estática en el centro)
star_position = np.array([0.0, 0.0, 0.0], dtype=np.float32)
star_color = np.array([1.0, 1.0, 0.8], dtype=np.float32)
star_size = 0.5

# Datos de las partículas
particles = []
max_particles = 5000
particle_lifetime = 120
particle_speed = 0.05
gravity = -0.005

# Variables para controlar la rotación
rotation_x = 0.0
rotation_y = 0.0
rotation_speed = 1.0

def generate_particle():
    # Nacer en el centro (ligeramente aleatorio para dispersión inicial)
    position = star_position + np.random.uniform(-0.1, 0.1, 3).astype(np.float32)
    # Velocidad inicial con componente hacia afuera y hacia arriba/abajo
    initial_velocity = np.array([
        np.random.uniform(-0.1, 0.1),
        random.uniform(0.1, 0.3),  # Componente vertical inicial
        np.random.uniform(-0.1, 0.1)
    ], dtype=np.float32) * particle_speed
    color = np.random.uniform(0.5, 1.0, 4).astype(np.float32)
    color[3] = 1.0
    return {
        'position': position,
        'velocity': initial_velocity,
        'lifetime': particle_lifetime,
        'color': color,
        'age': 0
    }

def update_particles():
    global particles
    new_particles = []
    num_new_particles = random.randint(5, 10)
    for _ in range(num_new_particles):
        if len(particles) < max_particles:
            particles.append(generate_particle())

    for p in particles:
        p['velocity'][1] += gravity  # Aplicar gravedad en la dirección Y
        p['position'] += p['velocity']
        p['lifetime'] -= 1
        p['age'] += 1
        if p['lifetime'] > 0:
            # Fade out las partículas al final de su vida
            alpha = p['lifetime'] / particle_lifetime
            new_p_color = np.array(p['color'])
            new_p_color[3] = alpha
            p['color'] = new_p_color
            new_particles.append(p)
    particles = new_particles

def render_star():
    glColor3fv(star_color)
    glPushMatrix()
    glTranslatef(*star_position)
    glutSolidSphere(star_size, 20, 20)  # Renderizar una esfera como estrella
    glPopMatrix()

def render_particles():
    glBegin(GL_POINTS)
    for p in particles:
        glColor4fv(p['color'])
        glVertex3fv(p['position'])
    glEnd()

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_LEFT:
                rotation_y += rotation_speed
            elif event.key == pygame.K_RIGHT:
                rotation_y -= rotation_speed
            elif event.key == pygame.K_UP:
                rotation_x += rotation_speed
            elif event.key == pygame.K_DOWN:
                rotation_x -= rotation_speed

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glLoadIdentity()
    glTranslatef(0.0, 0.0, -10.0)

    # Aplicar las rotaciones
    glRotatef(rotation_x, 1, 0, 0)
    glRotatef(rotation_y, 0, 1, 0)

    render_star()
    update_particles()
    render_particles()

    pygame.display.flip()
    time.sleep(0.01)

pygame.quit()
"""

In [ ]:
#Despliega una ventana con un cubo girando. es necesario dar permisos a display con "xhost +" en una consola en la maquina principal
codigo="""
import pygame
from pygame.locals import *

from OpenGL.GL import *
from OpenGL.GLU import *

verticies = (
    (1, -1, -1),
    (1, 1, -1),
    (-1, 1, -1),
    (-1, -1, -1),
    (1, -1, 1),
    (1, 1, 1),
    (-1, -1, 1),
    (-1, 1, 1)
    )

edges = (
    (0,1),
    (0,3),
    (0,4),
    (2,1),
    (2,3),
    (2,7),
    (6,3),
    (6,4),
    (6,7),
    (5,1),
    (5,4),
    (5,7)
    )


def Cube():
    glBegin(GL_LINES)
    for edge in edges:
        for vertex in edge:
            glVertex3fv(verticies[vertex])
    glEnd()


def main():
    pygame.init()
    display = (800,600)
    pygame.display.set_mode(display, DOUBLEBUF|OPENGL)

    gluPerspective(45, (display[0]/display[1]), 0.1, 50.0)

    glTranslatef(0.0,0.0, -5)

    while True:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                quit()

        glRotatef(1, 3, 1, 1)
        glClear(GL_COLOR_BUFFER_BIT|GL_DEPTH_BUFFER_BIT)
        Cube()
        pygame.display.flip()
        pygame.time.wait(10)


main()

"""

In [ ]:
#Carretero videos v1.0
codigo="""
import glfw
from OpenGL.GL import *
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import numpy as np
import os
import math
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
import sys

# Configuración
WIDTH, HEIGHT = 1920, 1080
# WIDTH, HEIGHT = 2560, 1440
# WIDTH, HEIGHT = 3840, 2160
# NUM_FRAMES = 200
FPS=30
# TEXT = "El Aegir cuarto, no era una nave famosa ni grandiosa."
FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
BACKGROUND_IMAGE = "./NucleoSilenteAudios/imagenes/capsulaAterrizandoR.png"
OUTPUT_TEMPLATE = "./frames13/frame_{:05d}.png"
LETTERS_PER_FRAME=2

def init_window():
	# Inicializar GLFW sin ventana visible
	glfw.init()
	glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
	glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 4)
	glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
	glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)
	window = glfw.create_window(WIDTH, HEIGHT, "Offscreen", None, None)
	glfw.make_context_current(window)

	# Imprimir información de OpenGL para verificar la GPU
	# print("--- Información de OpenGL (desde Docker) ---")
	# print("Vendor:", glGetString(GL_VENDOR).decode())
	# print("Renderer:", glGetString(GL_RENDERER).decode())
	# print("Version:", glGetString(GL_VERSION).decode())
	# print("GLSL Version:", glGetString(GL_SHADING_LANGUAGE_VERSION).decode())
	# print("------------------------------------------")
	
	return window

def wrap_text(text, font, max_width, draw):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        test_line = current_line + (" " if current_line else "") + word
        try:
            bbox = draw.textbbox((0, 0), test_line, font=font)
            width = bbox[2] - bbox[0]
        except AttributeError:
            width, _ = draw.textsize(test_line, font=font)

        if width <= max_width:
            current_line = test_line
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)
    return lines

def save_image(image, filename):
	image.save(filename)
	# print("✅ Guardado:", filename)

def create_blurred_background(image, target_width, target_height):
    # Escalar proporcionalmente (fit) y centrar la imagen sobre un fondo borroso
    image = image.convert("RGB")
    img_ratio = image.width / image.height
    target_ratio = target_width / target_height

    if img_ratio > target_ratio:
        scale = target_width / image.width
    else:
        scale = target_height / image.height

    # Escalar manteniendo relación
    scaled_size = (int(image.width * scale), int(image.height * scale))
    scaled_image = image.resize(scaled_size, Image.LANCZOS)

    # Crear fondo borroso del tamaño del viewport
    blurred_bg = image.resize((target_width, target_height), Image.LANCZOS).filter(ImageFilter.GaussianBlur(radius=30))

    # Pegar imagen escalada al centro
    offset = ((target_width - scaled_size[0]) // 2, (target_height - scaled_size[1]) // 2)
    blurred_bg.paste(scaled_image, offset)
    return blurred_bg

def create_blurred_background_moving(image, target_width, target_height, frame, speed=1.0):

	# Convertir a RGB
	image = image.convert("RGB")
	
	# Escalar para permitir desplazamiento
	SCALE = 1.1  # 110% para dejar margen
	large_width = int(target_width * SCALE)
	large_height = int(target_height * SCALE)
	blurred_base = image.resize((large_width, large_height), Image.LANCZOS)

	# Control del desplazamiento dinámico (con velocidad ajustable)
	max_shift_x = (large_width - target_width) // 2
	max_shift_y = (large_height - target_height) // 2

	# Movimiento oscilante con velocidad personalizada
	angle = 2 * math.pi * frame * speed / 360
	# angle = 2 * math.pi * frame * speed / total_frames 
	# print(f"{frame}\t{total_frames}\t{angle}")
	shift_x = int(max_shift_x * math.sin(angle))
	shift_y = int(max_shift_y * math.cos(angle))

	# Recortar área desplazada
	left = max_shift_x + shift_x
	top = max_shift_y + shift_y
	right = left + target_width
	bottom = top + target_height
	blurred_cropped = blurred_base.crop((left, top, right, bottom)).filter(ImageFilter.GaussianBlur(radius=30))

	# Escalar imagen central (nítida)
	img_ratio = image.width / image.height
	target_ratio = target_width / target_height
	if img_ratio > target_ratio:
		scale = target_width / image.width
	else:
		scale = target_height / image.height
	scaled_size = (int(image.width * scale), int(image.height * scale))
	scaled_image = image.resize(scaled_size, Image.LANCZOS)

	# Pegar centrada
	offset = ((target_width - scaled_size[0]) // 2, (target_height - scaled_size[1]) // 2)
	blurred_cropped.paste(scaled_image, offset)
	return blurred_cropped

# def generaSlide(segundos,TEXT):
def generaSlide(TEXT,frameInicial,frameFinal):
	
	NUM_FRAMES=frameFinal-frameInicial
	# print(f"Frame inicial {frameInicial}")
	# print(f"Frame final {frameFinal}")
	# print(f"NUM_FRAMES {NUM_FRAMES}")

	window = init_window()

	# Configurar viewport
	glViewport(0, 0, WIDTH, HEIGHT)

	# Cargar fondo y ajustarlo a tamaño
	# bg_image = Image.open(BACKGROUND_IMAGE).convert("RGB").resize((WIDTH, HEIGHT))

	# Cargar imagen de fondo con efecto blur
	original_image = Image.open(BACKGROUND_IMAGE)
	# bg_image = create_blurred_background(original_image, WIDTH, HEIGHT)
	
	# Preparar fuente
	try:
		font = ImageFont.truetype(FONT_PATH, 64)
	except OSError:
		font = ImageFont.load_default()

	#Para el paralelismo en la escritura de disco
	executor = ThreadPoolExecutor(max_workers=10)  # Puedes ajustar el número de hilos
	futures = []

	starttime=time.time()
	
	
	# Renderizar cada frame
	for frame in range(frameInicial,frameFinal):
		glClearColor(0.0, 0.0, 0.0, 1.0)
		glClear(GL_COLOR_BUFFER_BIT)

		bg_image = create_blurred_background_moving(original_image, WIDTH, HEIGHT, frame,0.5)

		# Leer del framebuffer OpenGL
		pixels = glReadPixels(0, 0, WIDTH, HEIGHT, GL_RGB, GL_UNSIGNED_BYTE)
		img = Image.frombytes("RGB", (WIDTH, HEIGHT), pixels)
		img = img.transpose(Image.FLIP_TOP_BOTTOM)

		# Componer sobre el fondo
		composite = Image.blend(bg_image, img, alpha=0.0)  # 100% fondo

		draw = ImageDraw.Draw(composite)
		
		# letters_to_show = min((frame + 1) * LETTERS_PER_FRAME, len(TEXT))
		# partial_text = TEXT[:letters_to_show]

		# # Calcular posición centrada
		# bbox = draw.textbbox((0, 0), TEXT, font=font)
		# text_width = bbox[2] - bbox[0]
		# text_height = bbox[3] - bbox[1]
		# x = (WIDTH - text_width) // 2
		# y = HEIGHT - text_height - 50
		# draw.text((x, y), partial_text, font=font, fill=(255, 255, 255))

		letters_to_show = min((frame-frameInicial + 1) * LETTERS_PER_FRAME, len(TEXT))
		partial_text = TEXT[:letters_to_show]

		# Crear objeto draw
		draw = ImageDraw.Draw(composite)

		## ComienzaDibuja texto
		# Ajustar texto a líneas (word wrap)
		linesComplete = wrap_text(TEXT, font, WIDTH - 40, draw)  # 40 px de margen
		lines = wrap_text(partial_text, font, WIDTH - 40, draw)  # 40 px de margen

		# Calcular altura de una línea
		try:
			bbox = draw.textbbox((0, 0), "Ag", font=font)
			line_height = bbox[3] - bbox[1]
		except:
			line_height = font.getsize("Ag")[1]

		# Posición vertical: desde abajo, con margen
		total_text_height = line_height * len(linesComplete)
		y_start = HEIGHT - total_text_height - 40  # 40 px de margen inferior

		# Dibujar cada línea centrada
		for i, line in enumerate(lines):
			try:
				text_width = draw.textbbox((0, 0), linesComplete[i], font=font)[2]
			except:
				text_width = draw.textsize(linesComplete[i], font=font)[0]
			x = (WIDTH - text_width) // 2
			y = y_start + i * line_height
			draw.text((x, y), line, font=font, fill=(255, 255, 255))
		## Dibuja texto


		output_filename = OUTPUT_TEMPLATE.format(frame)
		# output_filename = OUTPUT_TEMPLATE.format(frame)
		# composite.save(output_filename)
		# print("✅ Guardado:", output_filename)
		# Guardar imagen en segundo plano
		future = executor.submit(save_image, composite.copy(), output_filename)
		futures.append(future)

	nexttime=time.time()
	tiempogeneracion=nexttime-starttime
	# print(f"Frame final del for {frame}")
	# print(f"Finalizado en {nexttime-starttime}")
	# print(f"Esperando escritura en disco...")

	# Esperar a que todas las tareas de guardado terminen
	for future in futures:
		future.result()

	nexttime=time.time()
	print(f"✅Finalizado. #Frames:{NUM_FRAMES}, del {frameInicial} al {frameFinal}. TG:{tiempogeneracion:.02f} TT: {(nexttime-starttime):.02f}")

	glfw.terminate()

def main(args):
	# segundos=float(args[1])
	frameInicial=int(args[1])
	frameFinal=int(args[2])
	texto=args[3]
	# texto="El Aegir cuarto, no era una nave famosa ni grandiosa."
	# texto="En los bordes de la constelación de Vulpecula, más allá de los sistemas cartografiados, el Aegir cuarto detectó un objeto extraño:"

	generaSlide(texto,frameInicial,frameFinal)

if __name__ == "__main__":
	main(sys.argv)

"""

In [ ]:
#Carretero videos v1.0
codigo="""
import glfw
from OpenGL.GL import *
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import numpy as np
import os
import math
import time
from datetime import datetime
# from concurrent.futures import ThreadPoolExecutor
import sys
# import imageio.v3 as iio
import imageio_ffmpeg
from imageio_ffmpeg._io import (
    ffmpeg_test_encoder,
    get_compiled_h264_encoders,
    get_first_available_h264_encoder,
)
# from your_blur_module import blurred_background_horizontal, blurred_background_vertical, blurred_background_circular

# Configuración
WIDTH, HEIGHT = 1920, 1080
# WIDTH, HEIGHT = 2560, 1440 #Cuidar la división del ffmpeg
# WIDTH, HEIGHT = 3840, 2160
# NUM_FRAMES = 200

FPS=30
LETTERS_PER_FRAME=2

# TEXT = "El Aegir cuarto, no era una nave famosa ni grandiosa."
FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

RUTA="./NochesSanDamian"

# BACKGROUND_IMAGE = RUTA+"/imagenes/SanDamian.jpg"
# RUTA_IMAGE = RUTA+"/imagenes/AegirCuartoR2.png"
RUTA_IMAGE = RUTA+"/imagenes/"
LOGO = "./logoCarreterobigCircular.png"

PLAYGROUND="/Playground/"
# OUTPUT_TEMPLATE = RUTA+PLAYGROUND+"frame_{:05d}.png"
output_video_path=RUTA+PLAYGROUND+"frame_{:03d}.mp4"

def init_window():
	# Inicializar GLFW sin ventana visible
	glfw.init()
	glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
	glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 4)
	glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
	glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)
	window = glfw.create_window(WIDTH, HEIGHT, "Offscreen", None, None)
	glfw.make_context_current(window)
 
	# # Imprimir información de OpenGL para verificar la GPU
	# print("--- Información de OpenGL (desde Docker) ---")
	# print("Vendor:", glGetString(GL_VENDOR).decode())
	# print("Renderer:", glGetString(GL_RENDERER).decode())
	# print("Version:", glGetString(GL_VERSION).decode())
	# print("GLSL Version:", glGetString(GL_SHADING_LANGUAGE_VERSION).decode())
	# print("------------------------------------------")
	
	return window

def wrap_text(text, font, max_width, draw):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        test_line = current_line + (" " if current_line else "") + word
        try:
            bbox = draw.textbbox((0, 0), test_line, font=font)
            width = bbox[2] - bbox[0]
        except AttributeError:
            width, _ = draw.textsize(test_line, font=font)

        if width <= max_width:
            current_line = test_line
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)
    return lines

def save_image(image, filename):
	image.save(filename)
	# print("✅ Guardado:", filename)

def create_blurred_background(image, target_width, target_height):
    # Escalar proporcionalmente (fit) y centrar la imagen sobre un fondo borroso
    image = image.convert("RGB")
    img_ratio = image.width / image.height
    target_ratio = target_width / target_height

    if img_ratio > target_ratio:
        scale = target_width / image.width
    else:
        scale = target_height / image.height

    # Escalar manteniendo relación
    scaled_size = (int(image.width * scale), int(image.height * scale))
    scaled_image = image.resize(scaled_size, Image.LANCZOS)

    # Crear fondo borroso del tamaño del viewport
    blurred_bg = image.resize((target_width, target_height), Image.LANCZOS).filter(ImageFilter.GaussianBlur(radius=30))

    # Pegar imagen escalada al centro
    offset = ((target_width - scaled_size[0]) // 2, (target_height - scaled_size[1]) // 2)
    blurred_bg.paste(scaled_image, offset)
    return blurred_bg

def create_blurred_background_moving(image, centerimg,nextimg,target_width, target_height, frame, frameFinal,speed=1.0):

	image = image.convert("RGB")
	centerimg = centerimg.convert("RGBA")
	
	
	# Escalar para permitir desplazamiento
	SCALE = 1.1  # 110% para dejar margen
	large_width = int(target_width * SCALE)
	large_height = int(target_height * SCALE)
	blurred_base = image.resize((large_width, large_height), Image.LANCZOS)

	# Control del desplazamiento dinámico (con velocidad ajustable)
	max_shift_x = (large_width - target_width) // 2
	max_shift_y = (large_height - target_height) // 2

	# Movimiento oscilante con velocidad personalizada
	angle = 2 * math.pi * frame * speed / 360
	shift_x = int(max_shift_x * math.sin(angle))
	shift_y = int(max_shift_y * math.cos(angle))

	# Recortar área desplazada
	left = max_shift_x + shift_x
	top = max_shift_y + shift_y
	right = left + target_width
	bottom = top + target_height
	blurred_cropped = blurred_base.crop((left, top, right, bottom)).filter(ImageFilter.GaussianBlur(radius=30))

	def escalarimagen(img):
		# Escalar imagen central (nítida)
		img_ratio = img.width / img.height
		# target_ratio = target_width / target_height
		target_ratio = (target_height+int(target_height*0.2)) / target_height
		if img_ratio > target_ratio:
			# scale = target_width / image2.width
			scale = (target_height+int(target_height*0.2)) / img.width
		else:
			scale = target_height / img.height
		scaled_size = (int(img.width * scale), int(img.height * scale))
		return img.resize(scaled_size, Image.LANCZOS)
	
	centerImageScale=escalarimagen(centerimg)
	
	framesDesaparecer=50
	# Crear máscara de opacidad
	if frameFinal-frame < framesDesaparecer and nextimg != None:
		# fade_ratio = max(0, min(1, 1 - frame / frameFinal))  # De 1 a 0
		fade_ratio = (1 / framesDesaparecer)*(frameFinal-frame)  # De 1 a 0
		alpha = int(255 * fade_ratio)
		centerImageScale.putalpha(alpha)
		# print(f"Frame:{frame}, alpha:{alpha}, {frameFinal-frame},  fade_ratio:{fade_ratio}")
		nextimg = nextimg.convert("RGBA")
		nextImageScale=escalarimagen(nextimg)
		nextImageScale.putalpha(int(255 * (1-fade_ratio)))
		offset2 = ((target_width - nextImageScale.width) // 2, (target_height - nextImageScale.height) // 2)
		blurred_cropped.paste(nextImageScale, offset2,nextImageScale)
	
	offset = ((target_width - centerImageScale.width) // 2, (target_height - centerImageScale.height) // 2)
	blurred_cropped.paste(centerImageScale, offset,centerImageScale)

	return blurred_cropped

def create_background_moving(image, nextimage,target_width, target_height, frame, frameFinal,speed=1.0):

	image = image.convert("RGB")
	
	# Escalar para permitir desplazamiento
	SCALE = 1.1  # 110% para dejar margen
	large_width = int(target_width * SCALE)
	large_height = int(target_height * SCALE)
	blurred_base = image.resize((large_width, large_height), Image.LANCZOS)

	# Control del desplazamiento dinámico (con velocidad ajustable)
	max_shift_x = (large_width - target_width) // 2
	max_shift_y = (large_height - target_height) // 2

	# Movimiento oscilante con velocidad personalizada
	angle = 2 * math.pi * frame * speed / 360
	shift_x = int(max_shift_x * math.sin(angle))
	shift_y = int(max_shift_y * math.cos(angle))

	# Recortar área desplazada
	left = max_shift_x + shift_x
	top = max_shift_y + shift_y
	right = left + target_width
	bottom = top + target_height
	blurred_cropped = blurred_base.crop((left, top, right, bottom))

	return blurred_cropped

def create_background(image, nextimg, target_width, target_height, frame, frameFinal,speed=1.0):
	
	image = image.convert("RGB")
	
	# Escalar para permitir desplazamiento
	SCALE = 1.0  # 110% para dejar margen
	large_width = int(target_width * SCALE)
	large_height = int(target_height * SCALE)
	blurred_base = image.resize((large_width, large_height), Image.LANCZOS)

	framesDesaparecer=50
	# Crear máscara de opacidad
	if frameFinal-frame < framesDesaparecer and nextimg != None:
		# fade_ratio = max(0, min(1, 1 - frame / frameFinal))  # De 1 a 0
		fade_ratio = (1 / framesDesaparecer)*(frameFinal-frame)  # De 1 a 0
		alpha = int(255 * fade_ratio)
		# print(f"Frame:{frame}, alpha:{alpha}, {frameFinal-frame},  fade_ratio:{fade_ratio}")
		nextimg = nextimg.convert("RGBA")
		blurred_ni = nextimg.resize((large_width, large_height), Image.LANCZOS)
		blurred_ni.putalpha(int(255 * (1-fade_ratio)))
		offset2 = ((target_width - blurred_ni.width) // 2, (target_height - blurred_ni.height) // 2)
		blurred_base.paste(blurred_ni, offset2,blurred_ni)

	return blurred_base

# def generaSlide(segundos,TEXT):
def generaSlide(TEXT,frameInicial,frameFinal,secuencia,imagen,imgSiguiente):
	
	NUM_FRAMES=frameFinal-frameInicial
	# print(f"Frame inicial {frameInicial}")
	# print(f"Frame final {frameFinal}")
	# print(f"NUM_FRAMES {NUM_FRAMES}")

	window = init_window()

	# Configurar viewport
	glViewport(0, 0, WIDTH, HEIGHT)

	# Cargar imagen de fondo con efecto blur
	# background_image = Image.open(BACKGROUND_IMAGE)
	original_image = Image.open(RUTA_IMAGE+imagen)
	logo = Image.open(LOGO)

	if imagen != imgSiguiente:
		next_image = Image.open(RUTA_IMAGE+imgSiguiente)
	else:
		next_image=None

	# Preparar fuente
	try:
		# font = ImageFont.truetype(FONT_PATH, 64)
		font = ImageFont.truetype(FONT_PATH, int(HEIGHT*0.05))
		fontSombra = ImageFont.truetype(FONT_PATH, int(HEIGHT*0.05)+1)
	except OSError:
		font = ImageFont.load_default()

	# #Para el paralelismo en la escritura de disco
	# executor = ThreadPoolExecutor(max_workers=10)  # Puedes ajustar el número de hilos
	# futures = []

	starttime=time.time()
	
	draws=[]
	# Renderizar cada frame
	
	for frame in range(frameInicial,frameFinal):
		glClearColor(0.0, 0.0, 0.0, 1.0)
		glClear(GL_COLOR_BUFFER_BIT)

		# bg_image = create_blurred_background_moving(background_image,original_image, WIDTH, HEIGHT, frame,0.5)
		# bg_image = create_blurred_background_moving(background_image,original_image,next_image, WIDTH, HEIGHT, frame,frameFinal,0.5)
		# bg_image = create_background_moving(background_image, WIDTH, HEIGHT, frame,frameFinal,0.3)
		bg_image = create_background(original_image,next_image, WIDTH, HEIGHT, frame,frameFinal,0.3)

		#Insertando logo
		# scale=0.1
		# logo=logo.convert("RGBA")
		# scalelogo = logo.resize((int(logo.width*scale), int(logo.height*scale)), Image.LANCZOS)
		# offset = (WIDTH - int(WIDTH*0.05), HEIGHT - int(HEIGHT*0.05))
		# bg_image.paste(scalelogo, offset,scalelogo)

		scale=0.15
		logo=logo.convert("RGBA")
		scalelogo = logo.resize((int(logo.width*scale), int(logo.height*scale)), Image.LANCZOS)
		offset = (WIDTH - int(WIDTH*0.065), HEIGHT - int(HEIGHT*0.1))
		bg_image.paste(scalelogo, offset,scalelogo)

		# Leer del framebuffer OpenGL
		pixels = glReadPixels(0, 0, WIDTH, HEIGHT, GL_RGB, GL_UNSIGNED_BYTE)
		img = Image.frombytes("RGB", (WIDTH, HEIGHT), pixels)
		img = img.transpose(Image.FLIP_TOP_BOTTOM)

		# Componer sobre el fondo
		# composite = Image.blend(bg_image, img, alpha=0.0)  # 100% fondo
		composite = Image.blend(bg_image, img, alpha=0.0)  # 100% fondo

		letters_to_show = min((frame-frameInicial + 1) * LETTERS_PER_FRAME, len(TEXT))
		partial_text = TEXT[:letters_to_show]

		# Crear objeto draw
		draw = ImageDraw.Draw(composite)

		## ComienzaDibuja texto
		# Ajustar texto a líneas (word wrap)
		linesComplete = wrap_text(TEXT, font, WIDTH - 40, draw)  # 40 px de margen
		lines = wrap_text(partial_text, font, WIDTH - 40, draw)  # 40 px de margen

		# Calcular altura de una línea
		try:
			bbox = draw.textbbox((0, 0), "Ag", font=font)
			line_height = bbox[3] - bbox[1]
		except:
			line_height = font.getsize("Ag")[1]

		# Posición vertical: desde abajo, con margen
		total_text_height = line_height * len(linesComplete)
		y_start = HEIGHT - total_text_height - 40  # 40 px de margen inferior

		d=5
		# Dibujar cada línea centrada
		for i, line in enumerate(lines):
			try:
				text_width = draw.textbbox((0, 0), linesComplete[i], font=font)[2]
			except:
				text_width = draw.textsize(linesComplete[i], font=font)[0]
			x = (WIDTH - text_width) // 2
			y = y_start + i * line_height
			## Dibuja texto
			draw.text((x, y), line, font=font, fill=(255, 255, 255),stroke_width=6,stroke_fill=(10, 10, 10))
			draw.text((x, y), line, font=font, fill=(255, 255, 255),stroke_width=4,stroke_fill=(0, 0, 0))

		draws.append(composite)

		# output_filename = OUTPUT_TEMPLATE.format(frame)
		# # output_filename = OUTPUT_TEMPLATE.format(frame)
		# # composite.save(output_filename)
		# # print("✅ Guardado:", output_filename)
		# # Guardar imagen en segundo plano
		# future = executor.submit(save_image, composite.copy(), output_filename)
		# futures.append(future)


	nexttime=time.time()
	tiempogeneracion=nexttime-starttime
	# print(f"Frame final del for {frame}")
	# print(f"Finalizado en {tiempogeneracion}")
	# print(f"Esperando escritura en disco...")


	# Esperar a que todas las tareas de guardado terminen
	# for future in futures:
	# 	future.result()

	#https://github.com/imageio/imageio-ffmpeg
	gen = imageio_ffmpeg.write_frames(output_video_path.format(secuencia), (WIDTH, HEIGHT),fps=FPS,ffmpeg_log_level="error",macro_block_size=15)
	gen.send(None)  # seed the generator
	for frame in draws:
		gen.send(np.asarray(frame))
	gen.close()  # don't forget this

	# print("✅ Video guardado en:", output_video_path.format(secuencia))

	nexttime=time.time()
	print(f"✅ {secuencia:03d} finalizado.  del {frameInicial} al {frameFinal}. TG:{tiempogeneracion:.02f} TT: {(nexttime-starttime):.02f} {imagen} {imgSiguiente}")

	glfw.terminate()

def main(args):
	# segundos=float(args[1])
	frameInicial=int(args[1])
	frameFinal=int(args[2])
	texto=args[3]
	secuencia=int(args[4])
	imagen=args[5]
	imgSiguiente=args[6]
	if texto=="*silencio*":
		texto=" "
	generaSlide(texto,frameInicial,frameFinal,secuencia,imagen,imgSiguiente)

if __name__ == "__main__":
	main(sys.argv)
"""

In [38]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")
nameofpy="hipnosis"+formatted_time+".py"

with open(nameofpy, "w") as f:
    f.write(codigo)

In [3]:
!python3 {nameofpy}

python3: can't open file '/home/carretero/{nameofpy}': [Errno 2] No such file or directory


In [36]:
!ffmpeg -framerate 30 -y -i frames/frame_%04d.png -c:v libx264 -pix_fmt yuv420p hipnosis{formatted_time}.mp4
!rm -r frames

ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-libtwolame --enable-libvidstab --enab

In [ ]:
# Usando PIL para modificar una imagen
import PIL
print('PIL',PIL.__version__)

from PIL import Image
with Image.open("./NucleoSilente/imagenes/AegirCuartoR2.png") as im:
    im2 = im.copy()
    im2.putalpha(40)
    im.paste(im2, im)
    im.save("./NucleoSilente/imagenes/AegirCuartoR2_1.png")


## Generando codigo para mostrar oraciones con imagen detrás

In [ ]:
# !python3 CarreteroVideos.py
# !python3 CarreteroVideosSlides.py 3
# !python3 openglGemini.py
# !python3 ejemplo2OpenGLv3.3.py

#Con CPU
#Para 94 frames 20s
#Para 236 frames 57s

#Con GPU Creo que no ocupa la GPU
#Para 236 frames 1m1s


# !python3 render2ChatGPTOpenGLv3.3.py
# !rm frames13/*
!python3 CarreteroVids.py 3.1
#Para 200 frames 2m16s
#Para 200 frames 19s con paralelismo en la escritura
# from datetime import datetime
# now = datetime.now()
# formatted_time = now.strftime("%Y%m%d%H%M%S")
# ruta="frames13"
# !ffmpeg -framerate 30 -y -i {ruta}/frame_%05d.png -c:v libx264 -pix_fmt yuv420p {ruta}/avideo{formatted_time}.mp4
# print("✅✅ Video Creado")


Segundos 3.1

Frame inicial 180
Frame final 240
NUM_FRAMES 60
--- Información de OpenGL (desde Docker) ---
Vendor: NVIDIA Corporation
Renderer: NVIDIA GeForce RTX 3060/PCIe/SSE2
Version: 4.3.0 NVIDIA 550.144.03
GLSL Version: 4.30 NVIDIA via Cg compiler
------------------------------------------
Frame final del for 239
Finalizado en 12.784829378128052
Esperando escritura en disco...
✅ Escritura finalizada. Tiempo total: 13.186192750930786
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-li

In [124]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")
ruta="frames13"
!ffmpeg -framerate 30 -y -i {ruta}/frame_%05d.png -c:v libx264 -pix_fmt yuv420p {ruta}/avideo{formatted_time}.mp4

# !rm -r frames

#Unió 9968 frames en 1min21s

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
#Ejemplo para obtener los tiempos de todos los audios

import os
from pydub import AudioSegment

# Ruta a la carpeta que contiene los audios
carpetaAudios = "./NucleoSilenteAudios/"

# Lista para guardar la información (nombre, duración)
audios_info = []

# Recorremos todos los archivos en la carpeta
for archivo in os.listdir(carpetaAudios):
    if archivo.lower().endswith(".wav"):
        ruta_completa = os.path.join(carpetaAudios, archivo)
        audio = AudioSegment.from_wav(ruta_completa)
        duracion_segundos = len(audio) / 1000.0
        audios_info.append((archivo, duracion_segundos))

# Ordenar por nombre de archivo (alfabéticamente)
audios_info.sort(key=lambda x: x[0].lower())


# Mostramos los resultados
for nombre, duracion in audios_info:
    print(f"{nombre} -> {duracion:.2f} segundos")

00.2.wav -> 3.12 segundos
01.0.wav -> 3.77 segundos
02.0.wav -> 1.21 segundos
03.0.wav -> 7.84 segundos
04.0.wav -> 4.06 segundos
05.2.wav -> 2.67 segundos
06.0.wav -> 8.31 segundos
07.1.wav -> 2.72 segundos
08.0.wav -> 4.06 segundos
09.0.wav -> 3.83 segundos
10.0.wav -> 9.59 segundos
11.0.wav -> 4.24 segundos
12.0.wav -> 2.37 segundos
13.4.wav -> 4.99 segundos
14.0.wav -> 5.63 segundos
15.0.wav -> 3.36 segundos
16.0.wav -> 2.08 segundos
17.0.wav -> 3.77 segundos
18.1.wav -> 8.78 segundos
19.1.wav -> 2.08 segundos
20.0.wav -> 5.34 segundos
21.2.wav -> 7.43 segundos
22.0.wav -> 3.53 segundos
23.2.wav -> 4.59 segundos
24.0.wav -> 6.16 segundos
25.0.wav -> 4.12 segundos
26.0.wav -> 2.89 segundos
27.0.wav -> 3.36 segundos
28.0.wav -> 5.23 segundos
29.1.wav -> 5.98 segundos
29.6.wav -> 4.76 segundos
30.0.wav -> 6.68 segundos
31.0.wav -> 4.82 segundos
32.1.wav -> 2.37 segundos
33.3.wav -> 3.60 segundos
34.1.wav -> 2.77 segundos
35.2.wav -> 6.79 segundos
36.0.wav -> 5.28 segundos
37.0.wav -> 

In [ ]:
#Correr un archivo Python en paralelo
import subprocess
procs = []
for i in range(10):
# for i in range(21,40):
    # Pasa los argumentos como strings en la lista
    args = ["python3", "CarreteroVideosSlides.py", str(i), f"output_{i}.txt"]
    proc = subprocess.Popen(args)
    procs.append(proc)

# (Opcional) Esperar a que todos terminen
for proc in procs:
    proc.wait()

#Los primeros 4 en paralelo (480 frames) en 1m05s
#Los primeros 20 en praralelo (2663 frames) en 3min02s
#Los primeros 30 en praralelo (4126 frames) en 3min50s

#Con GPU ???? Creo que no se esta ocupando GPU con ese codigo
#Los primeros 10 en paralelo (1252 frames) en 1m42s


/usr/local/lib/python3.10/dist-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/usr/local/lib/p

Inicia 020
Finaliza 020
Inicia 052
Finaliza 052
Inicia 071
Finaliza 071
Inicia 002
Finaliza 002
Inicia 040
Finaliza 040
Inicia 010
Finaliza 010
Inicia 090
Finaliza 090
Inicia 080
Finaliza 080
Inicia 030
Finaliza 030
Inicia 060
Finaliza 060


In [ ]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")#frame_01000109.png
!ffmpeg -framerate 30 -y -i frames13/frame_%05d.png -c:v libx264 -pix_fmt yuv420p video{formatted_time}.mp4
# !rm -r frames

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
#Listar todos los archivos en un directorio
!find frames12/ -type f -name 'frame_*.png' | sort > ./archivos12.txt

In [50]:
import os

# Ruta de la carpeta donde están las imágenes
CARPETA = "./frames11"  # Cambia esto si las imágenes están en otra ruta

# Nombre del archivo de salida
OUTPUT_TXT = "input.txt"

# Duración de cada frame en segundos para 30 fps
FRAME_DURATION = 1 / 30

# Obtener y ordenar los archivos que coinciden
imagenes = sorted(f for f in os.listdir(CARPETA) if f.startswith("frame_") and f.endswith(".png"))

if not imagenes:
    raise ValueError("No se encontraron imágenes con patrón 'frame_*.png'.")

with open(OUTPUT_TXT, "w") as f:
    for imagen in imagenes:
        f.write(f"file '{CARPETA}/{imagen}'\n")
        f.write(f"duration {FRAME_DURATION:.4f}\n")
    # Agregar la última imagen una vez más (requisito de ffmpeg)
    f.write(f"file '{CARPETA}/{imagenes[-1]}'\n")

print(f"Archivo '{OUTPUT_TXT}' generado con {len(imagenes)} imágenes.")

Archivo 'input.txt' generado con 2663 imágenes.


In [51]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")#frame_01000109.png
!ffmpeg -f concat -safe 0 -i input.txt -c copy video{formatted_time}.mp4

ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-libtwolame --enable-libvidstab --enab

## Verdadero codigo corriendo Carretero Videos

In [1]:
import os
from pydub import AudioSegment
import subprocess

FPS=30

ruta= "./NochesSanDamian"
playgroundDir=ruta+"/Playground/"

os.makedirs(playgroundDir,exist_ok=True)

# try:
# 	os.mkdir(playgroundDir)
# except:
# 	print(playgroundDir,"Ya existe")

In [2]:
#Ejemplo para obtener los tiempos de todos los audios

# Ruta a la carpeta que contiene los audios
carpetaAudios = ruta+"/audios/"

# Lista para guardar la información (nombre, duración)
audios_info = []

# Recorremos todos los archivos en la carpeta
for archivo in os.listdir(carpetaAudios):
    if archivo.lower().endswith(".wav"):
        ruta_completa = os.path.join(carpetaAudios, archivo)
        audio = AudioSegment.from_wav(ruta_completa)
        duracion_segundos = len(audio) / 1000.0
        audios_info.append((archivo, duracion_segundos))

# Ordenar por nombre de archivo (alfabéticamente)
audios_info.sort(key=lambda x: x[0].lower())

# # Mostramos los resultados
# for nombre, duracion in audios_info:
#     print(f"{nombre} -> {duracion:.2f} segundos")

print(len(audios_info))

243


In [6]:
for nombre, duracion in audios_info:
    print(f"{nombre} -> {duracion:.2f} segundos")

0010.2.wav -> 6.96 segundos
0011.0.wav -> 7.67 segundos
0012.0.wav -> 5.28 segundos
0013.0.wav -> 4.24 segundos
0014.0.wav -> 10.58 segundos
0015.6.wav -> 9.00 segundos
0038.0.wav -> 2.01 segundos
0039.0.wav -> 2.01 segundos
0040.0.wav -> 6.16 segundos
0050.0.wav -> 6.16 segundos
0060.0.wav -> 3.31 segundos
0070.1.wav -> 8.31 segundos
0080.0.wav -> 3.60 segundos
0090.0.wav -> 9.12 segundos
0100.0.wav -> 7.15 segundos
0110.0.wav -> 8.54 segundos
0120.0.wav -> 7.50 segundos
0130.0.wav -> 4.59 segundos
0140.0.wav -> 4.41 segundos
0150.0.wav -> 3.01 segundos
0160.0.wav -> 3.83 segundos
0169.0.wav -> 2.01 segundos
0170.0.wav -> 3.48 segundos
0180.0.wav -> 4.29 segundos
0190.0.wav -> 2.25 segundos
0200.0.wav -> 9.36 segundos
0210.5.wav -> 4.99 segundos
0220.0.wav -> 3.60 segundos
0230.0.wav -> 7.32 segundos
0240.5.wav -> 6.86 segundos
0250.0.wav -> 1.49 segundos
0260.0.wav -> 5.57 segundos
0270.0.wav -> 2.43 segundos
0280.0.wav -> 3.65 segundos
0290.0.wav -> 4.93 segundos
0300.0.wav -> 8.14 

In [3]:
#Obteniendo Textos
nombre_archivo = ruta+"/textos.txt"

# Leer el archivo y almacenar cada línea en una lista
with open(nombre_archivo, 'r', encoding='utf-8') as archivo:
	lineas = [linea.strip() for linea in archivo]

# Mostrar el contenido del arreglo
print(len(lineas))
# print(lineas)


243


In [4]:
#Obteniendo imagenes
nombre_archivo = ruta+"/imagenes.txt"

# Leer el archivo y almacenar cada línea en una lista
with open(nombre_archivo, 'r', encoding='utf-8') as archivo:
	imagenes = [linea.strip() for linea in archivo]

# Mostrar el contenido del arreglo
print(len(imagenes))
# print(imagenes)

243


### Pruebas Individuales

In [40]:
i=0
# frameInicial=2654
# frameFinal=2814
frameInicial=0
frameFinal=200
texto=lineas[i]
print(texto)
# !rm frames/*
# !python3 CarreteroVidsv1.1.py {frameInicial} {frameFinal} "{texto}" 0 {"AegirCuartoR2.png"} {"planetoide.png"}
!python3 CarreteroVidsv1.1.py {frameInicial} {frameFinal} '{texto}' {i} {imagenes[i]} {imagenes[i+1]}
# !python3 CarreteroVidsv1.1.py {frameInicial} {frameFinal} "{texto}" {i} {"MuestrasdeshaciendoseR.png"} {"CuevadePielR.png"}

Hay caminos que todos los arrieros conocen, y hay pueblos donde ningún viajero sensato se queda después del anochecer.
✅ 000 finalizado.  del 0 al 200. TG:19.76 TT: 22.34 EusebiollegandoSanDamian.png EusebiollegandoSanDamian.png


In [ ]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")
ruta="frames13"
!ffmpeg -framerate 30 -y -i {ruta}/frame_%05d.png -c:v libx264 -pix_fmt yuv420p NucleoSilente{formatted_time}.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### Generación productiva

In [17]:
!rm {playgroundDir}*
!rm {playgroundDir}finales/*

rm: cannot remove './NochesSanDamian/Playground/finales': Is a directory


In [14]:
audioinfo=audios_info[0]
id=audioinfo[0].split(".")
print(id[0])

0010


In [15]:
rango=10
frameInicial=0
j=10
# for j in range(0,len(audios_info),rango):
procs = []
for i in range(j,min(j+rango,len(audios_info))):
	# print("-------")
	# print("frameInicial ",frameInicial)
	audioinfo=audios_info[i]
	id=audioinfo[0].split(".")
	# print(f"{i} - {audioinfo[0]} \t {audioinfo[1]} segundos, \t {(audioinfo[1]*FPS):.2f} \t {int(audioinfo[1]*FPS)}")
	# print(lineas[i])
	
	fframes=audioinfo[1]*FPS
	# frames=int(round(fframes)) if parte_decimal >= 0.5 else int(fframes)

	frames=round(fframes)
	frameFinal=frameInicial+int(frames)
	if not lineas[i].endswith(','):
		frames+=15
	# print("frameFinal ",frameFinal)
	# args = ["python3", "CarreteroVids.py", str(frameInicial),str(frameFinal),str(lineas[i]), f"{ruta}/output_{i}.txt"]
	# args = ["python3", "CarreteroVidsv1.1.py", str(frameInicial),str(frameFinal),str(lineas[i]),str(i), f"{ruta}/output_{i}.txt"]
	args = ["python3", "CarreteroVidsv1.1.py", str(frameInicial),str(frameFinal),str(lineas[i]),str(id[0]),str(imagenes[i]),str(imagenes[i if (i+1)>=len(audios_info) else i+1])]
	proc = subprocess.Popen(args)
	procs.append(proc)

	frameInicial=frameFinal

print(f"{j} - Todos los workers iniciando. Último frame: {frameFinal}")
# (Opcional) Esperar a que todos terminen
for proc in procs:
	proc.wait()

print(f"✅✅ {j:02d}s Finalizado")

# print(f"✅✅✅ Finalizado MUSIQUITA")

#Primeros 10 oraciones - 1242 frames - 01m59s
#Todas los 88 oraciones - 10930 frames - 16m34s CarreteroVids.py
#Todas los 88 oraciones - 10930 frames - 10m53s CarreteroVidsv1.1.py Generados a 16FPS
#Todas los 88 oraciones - 10930 frames - 10m53s CarreteroVidsv1.1.py Generados a 30FPS


10 - Todos los workers iniciando. Último frame: 1785
✅ 150 finalizado.  del 1695 al 1785. TG:14.67 TT: 17.62 llegandoSanDamian.png llegandoSanDamian.png
✅ 060 finalizado.  del 0 al 99. TG:16.93 TT: 21.24 llegandoSanDamian.png llegandoSanDamian.png
✅ 080 finalizado.  del 348 al 456. TG:17.91 TT: 21.98 llegandoSanDamian.png llegandoSanDamian.png
✅ 140 finalizado.  del 1563 al 1695. TG:24.30 TT: 28.55 llegandoSanDamian.png llegandoSanDamian.png
✅ 130 finalizado.  del 1425 al 1563. TG:24.43 TT: 28.74 llegandoSanDamian.png llegandoSanDamian.png
✅ 100 finalizado.  del 730 al 944. TG:36.10 TT: 39.96 llegandoSanDamian.png llegandoSanDamian.png
✅ 120 finalizado.  del 1200 al 1425. TG:38.37 TT: 42.58 llegandoSanDamian.png llegandoSanDamian.png
✅ 070 finalizado.  del 99 al 348. TG:42.87 TT: 48.70 llegandoSanDamian.png llegandoSanDamian.png
✅ 110 finalizado.  del 944 al 1200. TG:43.65 TT: 49.44 llegandoSanDamian.png llegandoSanDamian.png
✅ 090 finalizado.  del 456 al 730. TG:50.59 TT: 54.22 llegan

In [16]:
#Agregando audio a los videos generados

os.makedirs(ruta+"/Playground/finales/", exist_ok=True)
for i in range(len(audios_info)):
	nombre="{:03d}".format(i)
	# audioinfo=audios_info[i]
	comando = ["ffmpeg","-i", ruta+"/Playground/frame_"+nombre+".mp4","-i", carpetaAudios+"/"+audios_info[i][0],"-c:v", "copy","-c:a", "aac","-shortest",ruta+"/Playground/finales/final_"+nombre+".mp4"]
	proc = subprocess.Popen(comando)
	procs.append(proc)
	
for i,proc in enumerate(procs):
	proc.wait()
	print(f"Finalizado {i}")



ffmpeg version 5.1.6-0+deb12u1ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers Copyright (c) 2000-2024 the FFmpeg developers

  built with gcc 12 (Debian 12.2.0-14)
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex 

Finalizado 0
Finalizado 1
Finalizado 2
Finalizado 3
Finalizado 4
Finalizado 5
Finalizado 6
Finalizado 7
Finalizado 8
Finalizado 9
Finalizado 10
Finalizado 11
Finalizado 12
Finalizado 13
Finalizado 14
Finalizado 15
Finalizado 16
Finalizado 17
Finalizado 18
Finalizado 19
Finalizado 20
Finalizado 21
Finalizado 22
Finalizado 23
Finalizado 24
Finalizado 25
Finalizado 26
Finalizado 27
Finalizado 28
Finalizado 29
Finalizado 30
Finalizado 31
Finalizado 32
Finalizado 33
Finalizado 34
Finalizado 35
Finalizado 36
Finalizado 37
Finalizado 38
Finalizado 39
Finalizado 40
Finalizado 41
Finalizado 42
Finalizado 43
Finalizado 44
Finalizado 45
Finalizado 46
Finalizado 47
Finalizado 48
Finalizado 49
Finalizado 50
Finalizado 51
Finalizado 52
Finalizado 53
Finalizado 54
Finalizado 55
Finalizado 56
Finalizado 57
Finalizado 58
Finalizado 59
Finalizado 60
Finalizado 61
Finalizado 62
Finalizado 63
Finalizado 64
Finalizado 65
Finalizado 66
Finalizado 67
Finalizado 68
Finalizado 69
Finalizado 70
Finalizado 71
Fi

[aac @ 0x5cb6d7e18f40] Qavg: 584.533
ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-

Finalizado 110
Finalizado 111
Finalizado 112
Finalizado 113
Finalizado 114
Finalizado 115
Finalizado 116
Finalizado 117
Finalizado 118
Finalizado 119
Finalizado 120
Finalizado 121
Finalizado 122
Finalizado 123
Finalizado 124
Finalizado 125
Finalizado 126
Finalizado 127
Finalizado 128
Finalizado 129
Finalizado 130
Finalizado 131
Finalizado 132
Finalizado 133
Finalizado 134
Finalizado 135
Finalizado 136
Finalizado 137
Finalizado 138
Finalizado 139
Finalizado 140
Finalizado 141
Finalizado 142
Finalizado 143
Finalizado 144
Finalizado 145
Finalizado 146
Finalizado 147
Finalizado 148
Finalizado 149
Finalizado 150
Finalizado 151
Finalizado 152
Finalizado 153
Finalizado 154
Finalizado 155
Finalizado 156
Finalizado 157
Finalizado 158
Finalizado 159
Finalizado 160
Finalizado 161
Finalizado 162
Finalizado 163
Finalizado 164
Finalizado 165
Finalizado 166
Finalizado 167
Finalizado 168
Finalizado 169
Finalizado 170
Finalizado 171
Finalizado 172
Finalizado 173
Finalizado 174
Finalizado 175
Finalizado

In [ ]:
# Correr en consola para crear archivo de union
# printf "file '%s'\n" frame_*.mp4 | sort -V > lista_videos.txt
# printf "file '%s'\n" final_*.mp4 | sort -V > lista_videos.txt

In [23]:
# videosfinales=playgroundDir
videosfinales=playgroundDir+"/finales"
# Unir videos
!ffmpeg -f concat -safe 0 -i {videosfinales}/lista_videos.txt -c copy {videosfinales}/NochesSanDamianFinal.mp4

ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-libtwolame --enable-libvidstab --enab

In [149]:
!ffmpeg -i NucleoSilente20250525003001.mp4 -i NucleoSilenteAudios/NucleoSilenteEditadoCompleto.wav -c:v copy -c:a aac -shortest NucleoSilentev1.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [39]:
#Prueba para insertar logo en una imagen

from PIL import Image

bg_image = Image.open("./NochesSanDamian/imagenes/Adiossandamian.png")
logo = Image.open("./logoCarreterobigCircular.png")

WIDTH, HEIGHT = 1920, 1080

bg_image=bg_image.resize((WIDTH, HEIGHT), Image.LANCZOS)

scale=0.15
logo=logo.convert("RGBA")
scalelogo = logo.resize((int(logo.width*scale), int(logo.height*scale)), Image.LANCZOS)
# scalelogo.putalpha(50)
offset = (WIDTH - int(WIDTH*0.065), HEIGHT - int(HEIGHT*0.1))
bg_image.paste(scalelogo, offset,scalelogo)

bg_image.save("./NochesSanDamian/imagenes/pruebalogo.jpg")


